# Stage 1b: the checker again, with the right estimand and a held-out corpus

Stage one produced two findings. The checker recovered 30 of the 72 human successes and
failed its pre-set threshold, and the released `benchmark.py` did not reproduce the
released tasks: its T5 was a bare addition of two integers where the study's T5 is a
two-lookup average, and T0, T6 and T7 were written without their accents. The generator
is corrected; the nine surface forms used below are the ones the corpus actually holds.

Stage 1b changes two things about the method, and both corrections go against the way I
set the test up the first time.

**The second half of my threshold was the wrong criterion.** I required the checker to
recover nine in ten human successes *and* to reject nine in ten human failures. The
first is valid: the codebook makes a correct final answer necessary for S, so every
human S must be a checker success. The second is not. A human label names the first
observable fault, not the correctness of the answer, so a trajectory can contradict
itself mid-chain and still end on the right number. The pilot shows how far the two
estimands diverge: p02 and p15 answer their question correctly and are labelled F4,
p13 and p20 likewise and are labelled F2, p25 computes the right sum and is labelled
F1. Rejecting those would mean a *worse* checker, not a better one. The validation is
therefore one-directional, and the count of non-S trajectories called successful is
reported as a quantity to interpret rather than as an error rate.

**The answer-checking layer is developed on the pilot and then frozen.** Tuning it
against the 200 after inspecting the disagreements would make its agreement an in-sample
figure, which is the trap Section VI.C already flags for the recovered-detour exemption.
The split is explicit: layer 1 derives what a correct answer must contain from the task
text and the knowledge base, which is specified per corpus and uses no labels; layer 2
decides whether an answer contains it, and holds the parts that could be fitted, the
verbal-number parser, the numeric tolerance, the refusal vocabulary and the name
aliases. Layer 2 is tuned on the pilot's 13 successes alone, then applied unchanged to
the 200.

CPU only. No model is loaded.


In [ ]:
import ast, json, re, unicodedata, urllib.request
from collections import Counter

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")
fetch = lambda p: urllib.request.urlopen(BASE + p, timeout=90).read().decode("utf-8")

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines()
        if l.strip()]
pilot = [json.loads(l) for l in fetch("data/trajectories_pilot_30.jsonl").splitlines()
         if l.strip()]
_kb = json.loads(fetch("data/knowledge_base.json"))
KB, ABSENT = _kb["facts"], _kb["populations_absent_by_design"]
steps_of = lambda r: (ast.literal_eval(r["trajectory"])
                      if isinstance(r["trajectory"], str) else r["trajectory"])

POP = {k.split(" ", 1)[1]: int(v) for k, v in KB.items() if k.startswith("population ")}
CAP = {k.split(" ", 1)[1]: v for k, v in KB.items() if k.startswith("capitale ")}
KB_VALUES = {int(v) for v in KB.values() if str(v).isdigit()}

print("evaluation corpus %d | pilot %d | populations %d | capitals %d"
      % (len(rows), len(pilot), len(POP), len(CAP)))
print("pilot labels: %s" % dict(Counter(p["label_human"] for p in pilot).most_common()))


## Layer 2: deciding whether an answer contains what is expected

Four pieces, all of them developed on the pilot. Verbal magnitudes, because the agent
writes "13,96 millions d'habitants" as readily as a digit run. A relative tolerance of
one per cent, because a rounded "13,9 millions" against a stored 13 960 000 is a correct
answer to four significant figures and a wrong one to none. A refusal vocabulary, drawn
from the wordings the pilot actually produced. And an alias map, because the store holds
Alger and the agent sometimes writes Algiers.


In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s or "").lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9 ]+", " ", s)

MAG = {"millions": 1e6, "million": 1e6, "milliers": 1e3, "millier": 1e3,
       "milliards": 1e9, "milliard": 1e9}

def values(text):
    # Tous les nombres qu'une reponse enonce, les magnitudes en lettres comprises.
    s = str(text or "")
    out = set()
    for m in re.finditer(r"(\d[\d  .,]*\d|\d)\s*(millions?|milliers?|milliards?)",
                         s, re.I):
        raw, word = m.group(1), m.group(2).lower()
        raw = raw.replace(" ", "").replace(" ", "")
        if raw.count(",") == 1 and len(raw.split(",")[1]) <= 2:
            raw = raw.replace(".", "").replace(",", ".")
        else:
            raw = raw.replace(",", "").replace(".", "")
        try:
            out.add(float(raw) * MAG[word])
        except ValueError:
            pass
    for m in re.finditer(r"\d[\d  .,]{2,}\d|\d+", s):
        d = re.sub(r"\D", "", m.group())
        if d:
            out.add(float(int(d)))
    return out

def has_value(text, expected, rel=0.01):
    e = float(expected)
    for v in values(text):
        if abs(v - e) <= max(1.0, rel * abs(e)):
            return True
    return False

REFUSAL = ("non trouve", "pas trouve", "pas pu trouver", "introuvab",
           "non disponible", "pas disponible", "indisponible", "ne figure pas",
           "aucune information", "aucune donnee", "pas dans la base", "inconnu",
           "impossible", "je ne peux pas", "pas de donnee", "information non",
           "n est pas accessible", "pas accessible", "ne peut etre",
           "ne peut pas etre", "je suis desole", "sans plus de contexte",
           "pas ete capable", "ne peux pas fournir", "ne peut pas etre determinee")

def refuses(text):
    t = norm(text)
    return any(k in t for k in REFUSAL)

ALIAS = {"algiers": "alger", "brasilia": "brasilia", "le caire": "le caire",
         "new delhi": "new delhi", "tokio": "tokyo", "rome": "rome"}

def has_text(answer, expected):
    a, e = norm(answer), norm(expected)
    if e in a:
        return True
    for k, v in ALIAS.items():
        if norm(v) == e and k in a:
            return True
    return False

def final_answer(steps):
    for s in reversed(steps):
        if str(s.get("action")).strip().lower() == "finish":
            v = s.get("input")
            return "" if v is None or str(v).strip() in ("", "None") else str(v)
    return ""

def verdict(answer, kind, payload):
    # (bool, raison). kind vient de la couche 1.
    if not str(answer).strip():
        return False, "empty final answer"
    if kind == "text":
        return has_text(answer, payload), "expects the text %r" % payload
    if kind == "number":
        return has_value(answer, payload), "expects %s" % format(payload, ".0f")
    if kind == "refusal":
        if not refuses(answer):
            return False, "no refusal where the value is absent from the store"
        invented = {v for v in values(answer)
                    if v >= 100000 and not any(abs(v - k) <= max(1.0, 0.01 * k)
                                               for k in KB_VALUES)}
        if invented:
            return False, ("refusal wording with invented value(s) %s"
                           % sorted(int(v) for v in invented))
        return True, "refusal with no invented value"
    if kind == "capital_no_population":
        if not has_text(answer, payload):
            return False, "capital %r absent" % payload
        asserted = {v for v in values(answer) if v >= 10000}
        if asserted and not refuses(answer):
            return False, ("a country population is asserted: %s"
                           % sorted(int(v) for v in asserted))
        return True, "capital given, no country population invented"
    if kind == "city_and_gap":
        city, gap = payload
        if not has_text(answer, city):
            return False, "the more populous city %r is not named" % city
        return has_value(answer, gap), "expects the gap %d" % gap
    return False, "not derivable"

print("layer 2 defined: %d refusal markers, %d aliases, tolerance 1 per cent"
      % (len(REFUSAL), len(ALIAS)))


## The pilot, as the development set

The expectation for each pilot task is derived here from the task text and the store,
and from nothing else. Four of the thirty are not derivable from the store at all and
are marked as such rather than guessed: the largest city in the world is an open-world
question, and the three tasks whose answer the store cannot supply are refusals only if
the store is the only admissible source, which for a named city outside it is the case
and is stated per row.

The test is one-directional: every pilot trajectory the human labelled S must come out
as a success. Nothing is tuned on the other seventeen.


In [ ]:
P = POP
EXP = {
 "p00": ("text", CAP["france"]),
 "p01": ("text", CAP["japon"]),
 "p02": ("text", CAP["algerie"]),
 "p03": ("number", P["paris"]),
 "p04": ("number", P["tokyo"]),
 "p05": ("number", P["alger"]),
 "p06": ("number", 12 * 7),
 "p07": ("number", 158 + 247),
 "p08": ("number", 1000 / 25),
 "p09": ("number", 50 + 30 - 15),
 "p10": ("number", P["paris"] / 1000),
 "p11": ("number", P["tokyo"] - P["alger"]),
 "p12": ("number", P["alger"] * 2),
 "p13": ("number", P["paris"] + P["alger"]),
 "p14": ("number", P["tokyo"] / 2),
 "p15": ("text", CAP["bresil"]),
 "p16": ("refusal", None),
 "p17": ("text", CAP["espagne"]),
 "p18": ("number", 0.15 * 200),
 "p19": ("number", 2 ** 10),
 "p20": ("number", 12),
 "p21": ("text", CAP["france"]),
 "p22": ("number", P["tokyo"]),
 "p23": ("number", P["paris"]),
 "p24": ("not derivable", None),
 "p25": ("number", P["paris"] + P["tokyo"]),
 "p26": ("number", P["alger"] * 2),
 "p27": ("number", P["tokyo"] - P["paris"]),
 "p28": ("number", 1234 + 5678),
 "p29": ("capital_no_population", CAP["france"]),
}
missing = [p["id"] for p in pilot if p["id"] not in EXP]
print("pilot rows without an expectation: %s" % (missing or "none"))

ok_s, bad_s, skipped = 0, [], 0
for p in pilot:
    kind, payload = EXP[p["id"]]
    ans = final_answer(steps_of(p))
    if kind == "not derivable":
        skipped += 1
        continue
    good, why = verdict(ans, kind, payload)
    if p["label_human"] == "S":
        if good:
            ok_s += 1
        else:
            bad_s.append((p["id"], p["task"], ans, why))

n_s = sum(1 for p in pilot if p["label_human"] == "S"
          and EXP[p["id"]][0] != "not derivable")
print("pilot successes covered by an expectation: %d" % n_s)
print("recovered by the checker: %d (%.1f%%)" % (ok_s, 100.0 * ok_s / max(1, n_s)))
print("not derivable, excluded: %d" % skipped)
print()
for pid, task, ans, why in bad_s:
    print("MISS %s | %s" % (pid, task[:70]))
    print("     answer : %s" % (ans[:90] or "(empty)"))
    print("     reason : %s" % why)


In [ ]:
print("=== for reference only, the other pilot rows ===")
print("%-5s %-4s %-9s %s" % ("id", "hum", "checker", "answer"))
for p in pilot:
    kind, payload = EXP[p["id"]]
    if kind == "not derivable":
        print("%-5s %-4s %-9s %s" % (p["id"], p["label_human"], "n/a",
                                     "(no expectation derivable)"))
        continue
    ans = final_answer(steps_of(p))
    good, _ = verdict(ans, kind, payload)
    print("%-5s %-4s %-9s %s" % (p["id"], p["label_human"],
                                 "success" if good else "failure", ans[:58] or "(empty)"))


## The 200, held out

Layer 1 for this corpus comes from the nine surface forms the corrected generator
declares. Layer 2 is whatever the pilot left it as, unchanged.


In [ ]:
FORMS = {
 "T0_capital": r"^Quelle est la capitale de (\w+) \?$",
 "T0_product": r"^Combien font (\d+) multiplié par (\d+) \?$",
 "T1": r"^Calcule la population de (\w+)\.$",
 "T2": r"^Quelle est la population de (\w+) \?$",
 "T3": r"^Donne la capitale et la population de (\w+)\.$",
 "T4": r"^Trouve absolument la population de (\w+), c'est important\.$",
 "T5": (r"^Additionne les populations de (\w+) et (\w+), puis divise le résultat "
        r"par 2\.$"),
 "T6": (r"^Quelle ville est la plus peuplée, (\w+) ou (\w+), et quel est l'écart "
        r"de population \?$"),
 "T7": r"^Quelle est la différence de population entre (\w+) et (\w+) \?$",
}
RX = {k: re.compile(v) for k, v in FORMS.items()}

def expectation(task):
    for name, rx in RX.items():
        m = rx.match(task)
        if not m:
            continue
        g = m.groups()
        if name == "T0_capital":
            return ("text", CAP.get(g[0]))
        if name == "T0_product":
            return ("number", int(g[0]) * int(g[1]))
        if name == "T1":
            return ("number", POP.get(g[0]))
        if name in ("T2", "T4", "T7"):
            return ("refusal", None)
        if name == "T3":
            return ("capital_no_population", CAP.get(g[0]))
        if name == "T5":
            if g[0] in POP and g[1] in POP:
                return ("number", (POP[g[0]] + POP[g[1]]) / 2)
            return ("not derivable", None)
        if name == "T6":
            if g[0] in POP and g[1] in POP:
                big = g[0] if POP[g[0]] >= POP[g[1]] else g[1]
                return ("city_and_gap", (big, abs(POP[g[0]] - POP[g[1]])))
            return ("not derivable", None)
    return ("not derivable", None)

nd = [r["id"] for r in rows if expectation(r["task"])[0] == "not derivable"]
print("evaluation tasks with no derivable expectation: %d %s" % (len(nd), nd[:5]))

pred, why = [], []
for r in rows:
    kind, payload = expectation(r["task"])
    good, reason = verdict(final_answer(steps_of(r)), kind, payload)
    pred.append(good)
    why.append(reason)
human = [r["label_human"] == "S" for r in rows]
print("checker successes on the 200: %d | human successes: %d"
      % (sum(pred), sum(human)))


In [ ]:
tp = sum(1 for h, p in zip(human, pred) if h and p)
fn = sum(1 for h, p in zip(human, pred) if h and not p)
fp = sum(1 for h, p in zip(human, pred) if not h and p)
rec = tp / max(1, sum(human))
print("=== the valid direction: every human success must be a checker success ===")
print("recovered %d of %d human successes = %.3f (threshold 0.90)"
      % (tp, sum(human), rec))
print()
print("the %d human successes the checker still rejects:" % fn)
for i, r in enumerate(rows):
    if human[i] and not pred[i]:
        print("   %s | %-3s | %s" % (r["id"], r["template"], why[i]))
        print("        answer : %s" % (final_answer(steps_of(r))[:88] or "(empty)"))
print()
print("=== the other direction, to interpret and not to score ===")
print("%d trajectories the human labelled a failure end on a correct answer:" % fp)
print("   by human label: %s"
      % dict(Counter(rows[i]["label_human"] for i in range(len(rows))
                     if pred[i] and not human[i]).most_common()))
print("   by template   : %s"
      % dict(Counter(rows[i]["template"] for i in range(len(rows))
                     if pred[i] and not human[i]).most_common()))


In [ ]:
print("%-5s %7s %9s %10s %s" % ("tmpl", "n", "human S", "checker S", "human S kept"))
for tpl in sorted({r["template"] for r in rows}):
    idx = [i for i, r in enumerate(rows) if r["template"] == tpl]
    hs = [i for i in idx if human[i]]
    kept = sum(1 for i in hs if pred[i])
    print("%-5s %7d %9d %10d %7d/%d"
          % (tpl, len(idx), len(hs), sum(1 for i in idx if pred[i]), kept, len(hs)))
print()
passes = rec >= 0.90
print("VERDICT: %s" % ("the checker can carry the intervention experiment"
                       if passes else "not yet good enough; the misses are listed above"))
json.dump({"pilot_recovered": ok_s, "pilot_successes": n_s,
           "held_out_recovered": tp, "held_out_successes": sum(human),
           "recovery": round(rec, 3), "correct_answer_despite_fault": fp,
           "passes": bool(passes), "pred": pred},
          open("success_checker_v2.json", "w"))
print("written: success_checker_v2.json")
